In [1]:
# load env
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from pathlib import Path
import os
import re
from pprint import pprint

from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path.cwd()

# If your notebook opens inside /notebooks, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PAPER_DIR = PROJECT_ROOT / "paper"
INDEX_DIR = PROJECT_ROOT / "storage" / "faiss_index"

print("Project root:", PROJECT_ROOT)
print("Paper folder:", PAPER_DIR)
print("Index folder:", INDEX_DIR)
print("Paper folder exists?", PAPER_DIR.exists())
print("OpenAI key loaded?", bool(os.getenv("OPENAI_API_KEY")))

Project root: c:\Tutorial\rag\RagTeX
Paper folder: c:\Tutorial\rag\RagTeX\paper
Index folder: c:\Tutorial\rag\RagTeX\storage\faiss_index
Paper folder exists? True
OpenAI key loaded? True


In [3]:
for path in sorted(PAPER_DIR.iterdir()):
    print(path.name)

Appendix.tex
having the graph.tex
HMC.bib
Images
main.tex
notzot.bib
numerical_images
old_writeups
UF_FRED_paper_style.sty


In [5]:
EXCLUDED_DIR_NAMES = {
    "Images",
    "images",
    "numerical_images",
    "old_writeups",
    ".git",
    "having the graph.tex",
    "__pycache__",
}

def should_skip_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)

tex_files = sorted(
    path for path in PAPER_DIR.rglob("*.tex")
    if path.is_file() and not should_skip_path(path)
)

print(f"Found {len(tex_files)} .tex files:\n")
for path in tex_files:
    print("-", path.relative_to(PAPER_DIR))

Found 2 .tex files:

- Appendix.tex
- main.tex


In [6]:
test_file = PAPER_DIR / "main.tex"
print("\nTesting file content:\n")
raw_content = test_file.read_text(encoding="utf-8")
print(raw_content[:500])


Testing file content:

\documentclass[12pt]{article}
\usepackage{UF_FRED_paper_style}
\usepackage{amsmath, amssymb, amsthm}
\usepackage{geometry}
\usepackage{enumitem}
\usepackage{natbib}
\bibliographystyle{plainnat}
\doublespacing
% aviod breaking the words by - ------
\usepackage{microtype} % helps justification a lot

\hyphenpenalty=8000        % discourage hyphenation (0–10000)
\exhyphenpenalty=2000      % allow breaks at explicit hyphens if needed
\emergencystretch=2em      % last-resort stretch to avoid overflow


In [7]:
def strip_latex_comments(text: str) -> str:
    """
    Remove LaTeX comments.

    A good starting point for cleaning
    """
    cleaned_lines = []

    for line in text.splitlines():
        cleaned_line = re.sub(r"(?<!\\)%.*$", "", line).rstrip()
        if cleaned_line:
            cleaned_lines.append(cleaned_line)

    return "\n".join(cleaned_lines)

In [8]:
cleaned_text = strip_latex_comments(raw_content)

print("Raw characters:", len(raw_content))
print("Cleaned characters:", len(cleaned_text))
print(cleaned_text[:500])

Raw characters: 137990
Cleaned characters: 124307
\documentclass[12pt]{article}
\usepackage{UF_FRED_paper_style}
\usepackage{amsmath, amssymb, amsthm}
\usepackage{geometry}
\usepackage{enumitem}
\usepackage{natbib}
\bibliographystyle{plainnat}
\doublespacing
\usepackage{microtype}
\hyphenpenalty=8000
\exhyphenpenalty=2000
\emergencystretch=2em
\usepackage{changes}
\usepackage{comment}
\usepackage{xcolor}
\newcommand{\note}[1]{\noindent\textit{\textcolor{blue}{#1}}}
\usepackage{tikz}
\usetikzlibrary{decorations.pathreplacing,calc}
\usetikzlibrar


In [9]:
from langchain_core.documents import Document

documents = []

for path in tex_files:
    raw = path.read_text(encoding="utf-8", errors="ignore")
    cleaned = strip_latex_comments(raw)

    doc = Document(
        page_content=cleaned,
        metadata={
            "source": str(path.relative_to(PAPER_DIR)),
            "file_name": path.name,
            "file_type": ".tex",
        },
    )

    documents.append(doc)

In [10]:
print(f"Created {len(documents)} LangChain documents.\n")

for doc in documents:
    print(doc.metadata, "characters:", len(doc.page_content))

Created 2 LangChain documents.

{'source': 'Appendix.tex', 'file_name': 'Appendix.tex', 'file_type': '.tex'} characters: 64271
{'source': 'main.tex', 'file_name': 'main.tex', 'file_type': '.tex'} characters: 124307


## Chunk the paper

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.LATEX,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = splitter.split_documents(documents)
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i


In [20]:
print("Number of original documents:", len(documents))
print("Number of chunks:", len(chunks))

Number of original documents: 2
Number of chunks: 202


In [21]:
chunks[145:148]

[Document(metadata={'source': 'main.tex', 'file_name': 'main.tex', 'file_type': '.tex', 'chunk_id': 145}, page_content='(0,0.12) -- (\\ug,0.12)\n  node[midway,above=8pt] {$F=0$};\n\\draw[decorate,decoration={brace,amplitude=6pt}]\n  (\\ug,0.12) -- (\\mua,0.12)\n  node[midway,above=8pt] {$F<0$};\n\\draw[decorate,decoration={brace,amplitude=6pt}]\n  (\\mua,0.12) -- (\\ub,0.12)\n  node[midway,above=8pt] {$F>0$};\n\\draw[decorate,decoration={brace,amplitude=6pt}]\n  (\\ub,0.12) -- (\\og,0.12)\n  node[midway,above=8pt] {$F=0$};\n\\draw[decorate,decoration={brace,amplitude=6pt}]\n  (\\og,0.12) -- (\\mub,0.12)\n  node[midway,above=8pt] {$F>0$};\n\\draw[decorate,decoration={brace,amplitude=6pt}]\n  (\\mub,0.12) -- (\\ob,0.12)\n  node[midway,above=8pt] {$F<0$};\n\\draw[decorate,decoration={brace,amplitude=6pt}]\n  (\\ob,0.12) -- (1,0.12)\n  node[midway,above=8pt] {$F=0$};\n\\end{tikzpicture}\n\\vspace{0.5em}\n{\\small (b) $\\underline{\\mu}^{b}_{0} < \\overline{\\mu}^{g}_{1}$: a fair region in 

In [22]:
for chunk in chunks[145:148]:
    print(len(chunk.page_content))

1163
1197
1194
